# Notebook de Transformação — MOD_SINT_001
## DVA-CBS | TC 015.848/2025-6

**Objetivo:** Cruzamento das bases EFD+CNPJ+RAIS, flagging de outliers e geração da planilha consolidada.

**Executor:** SERPRO/COSIT | **Data:** 2026-02-14

In [ ]:
import pandas as pd
import numpy as np

# Carregamento dos dados (output do script_extracao.sql)
df = pd.read_csv('consolidado_anual_raw.csv', dtype={'cnpj_raiz': str})
print(f'Registros carregados: {len(df):,}')
print(f'CNPJs únicos: {df["cnpj_raiz"].nunique():,}')

In [ ]:
# Cruzamento com RAIS 2025
rais = pd.read_csv('rais_2025_ti.csv', dtype={'cnpj_raiz': str})
df = df.merge(rais[['cnpj_raiz', 'total_vinculos', 'salario_medio']], 
              on='cnpj_raiz', how='left')
df['flag_rais_ausente'] = df['total_vinculos'].isna().astype(int)

print(f'Empresas sem RAIS: {df["flag_rais_ausente"].sum():,}')
# Esperado: 412 conforme descricao_metodologica.md

In [ ]:
# Identificação de outliers: ±3 desvios-padrão
media = df['aliquota_efetiva_anual'].mean()
desvio = df['aliquota_efetiva_anual'].std()

df['z_score'] = (df['aliquota_efetiva_anual'] - media).abs() / desvio
df['flag_outlier'] = (df['z_score'] > 3).astype(int)

n_outliers = df['flag_outlier'].sum()
print(f'Outliers identificados: {n_outliers}')
# Esperado: 147 conforme descricao_metodologica.md

In [ ]:
# Cálculo da alíquota de referência (excluindo outliers)
df_limpo = df[df['flag_outlier'] == 0].copy()

resultado = df_limpo.groupby('porte_empresa').agg(
    qtd_empresas=('cnpj_raiz', 'nunique'),
    bc_total=('bc_total_anual', 'sum'),
    cbs_total=('cbs_total_anual', 'sum')
).reset_index()

resultado['aliquota_ref_pct'] = (
    resultado['cbs_total'] / resultado['bc_total'] * 100
).round(2)

print(resultado.to_string(index=False))

In [ ]:
# VERIFICAÇÃO DE CONSISTÊNCIA — resultado deve bater com script_extracao.sql
aliquota_total = resultado['cbs_total'].sum() / resultado['bc_total'].sum() * 100
print(f'Alíquota de referência total: {aliquota_total:.2f}%')
assert abs(aliquota_total - 8.77) < 0.01, f'INCONSISTÊNCIA: esperado 8.77%, obtido {aliquota_total:.2f}%'
print('✓ Consistência verificada: alíquota bate com script SQL')

n_total = resultado['qtd_empresas'].sum()
assert n_total == 88487, f'INCONSISTÊNCIA: esperado 88.487 empresas, obtido {n_total}'
print('✓ Consistência verificada: contagem de CNPJs bate com script SQL')